# A tiny GPT writes Shakespeare

*ai · intermediate* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/ai/tiny_transformer)

A character-level GPT built from raw tensor ops: embeddings, causal multi-head self-attention written as einsum, LayerNorm, an MLP and residual connections, no nn.Transformer. Trained for about a minute on Macbeth and Romeo and Juliet, its loss falls from $\ln V$ (a uniform guess over the $V$ characters) and its samples turn from gibberish into words, names and verse-shaped lines, while the attention maps stay strictly lower-triangular.

**Vector form**

$$\mathbf A^{(h)} = \operatorname{softmax}\!\Big(\frac{\mathbf Q^{(h)}\mathbf K^{(h)\mathsf T}}{\sqrt{d_k}} + \mathbf M\Big),\qquad M_{ij} = \begin{cases}0 & j \le i\ -\infty & j > i\end{cases},\qquad \mathcal L = -\frac1{NT}\sum_{n,i}\log p_\theta(t_{n,i+1}\mid t_{n,\le i})$$

**Index (tensor) form**

$$Q_{nhik} = \hat X_{nic}W^Q_{hck} \qquad A_{nhij} = \mathrm{softmax}_j\big(Q_{nhik}K_{nhjk}/\sqrt{d_k} + M_{ij}\big) \qquad X_{nic} \mathrel{+}= A_{nhij}V_{nhjk}W^O_{hkc}$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $t_{ni}$ | `batch(data, n, ctx, gen)` | (N, T) | random windows of the text as character ids; the targets are the same windows shifted one character left |
| $X_{nic}$ | `w['tok'][tokens] + w['pos'][:T]` | (N, T, d) | token plus learned position embedding |
| $\hat X$ | `layer_norm(X, g, b)` | (N, T, d) | every token vector shifted to mean 0 and scaled to variance 1 over its channels, then a learned scale and shift |
| $Q_{nhik}$ | `einsum('nic,hck->nhik', X, Wq)` | (N, H, T, d_k) | repeated $c$ ⇒ summed: the queries of every head (keys and values alike) |
| $M_{ij}$ | `causal_mask(T, device)` | (T, T) | $0$ on and below the diagonal, $-\infty$ above it: after the softmax a position cannot see later characters |
| $A_{nhij}$ | `softmax(einsum('nhik,nhjk->nhij', Q, K) / sqrt(dk) + M, -1)` | (N, H, T, T) | repeated $k$ ⇒ summed; each row is a probability distribution over the earlier positions $j \le i$ |
| $A_{nhij}V_{nhjk}W^O_{hkc}$ | `einsum('nhik,hkc->nic', A @ V, Wo)` | (N, T, d) | repeated $j$, then $h, k$ ⇒ summed: what each position gathers from its past, added back through the residual |
| $\mathcal L$ | `cross_entropy(logits, targets)` | () | mean $-\log p$ of the true next character in nats; it starts at $\ln V$ |

The architecture is that of Vaswani et al., "Attention is all you need" (2017), in its decoder-only GPT form with pre-norm blocks: $X \leftarrow X + \mathrm{Attn}(\mathrm{LN}\,X)$, then $X \leftarrow X + \mathrm{MLP}(\mathrm{LN}\,X)$. The tests check the mechanics exactly: the einsum attention equals PyTorch's fused `scaled_dot_product_attention` with `is_causal=True`; changing characters after position $t$ leaves every output up to $t$ unchanged (the mask); rows of $A$ sum to 1; the untrained loss is $\ln V$; the model can memorise one batch; and the weight count is $Vd + Td + L(12d^2 + 13d) + 2d + dV + V$. Because of the mask, one forward pass trains all $T$ next-character predictions of a window at once, while generation must run one character at a time. A model this small (113 093 weights, about a minute of CPU) learns spelling, spacing, capitalised speaker names and line breaks, not meaning; the validation loss ends near 2 nats per character. The heatmaps show, for each layer, the head that attends most to the previous character on the line "When shall we three meet again?". The bioinformatics protein language model uses the same attention without a mask, to fill in hidden letters. Text: Macbeth and Romeo and Juliet from The Complete Works of William Shakespeare, Project Gutenberg eBook #100 (public domain), with the eBook header and licence removed, curly quotes made ASCII; 247 kB in data/shakespeare.txt. Try more steps, a wider model, or temperature 0.5 versus 1.2.


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/ai/tiny_transformer/main.py
"""Main loop: Shakespeare → character tokens → hand-written GPT → Adam on next-character loss → samples and plot."""

import math
from pathlib import Path

import torch

from problems._common.runtime import cli, device, fast_mode

from . import plot, solve, tensors

EXCERPT = "When shall we three meet again?"  # Macbeth, Act I, Scene I: the line whose attention is drawn


def run(params: dict, out_dir: Path) -> Path:
    dev = device()
    fast = fast_mode()
    text = tensors.load_text()
    chars, data = tensors.encode_text(text, dev)
    split = int(0.9 * len(data))
    train_data, val_data = data[:split], data[split:]
    V = len(chars)
    print(f"text: {len(text)} characters of Macbeth and Romeo and Juliet, vocabulary V = {V}, ln V = {math.log(V):.3f}")

    d, heads, layers, ctx = params["d_model"], params["heads"], params["layers"], params["context"]
    w = tensors.init_weights(V, d, heads, layers, ctx, seed=0, device=dev)
    n_params = sum(t.numel() for t in tensors.parameters(w))
    print(f"GPT: {layers} layers × {heads} heads, d = {d}, context T = {ctx}: {n_params} weights "
          f"(formula {tensors.parameter_count(V, d, layers, ctx)})")  # fmt: skip

    n_steps = 20 if fast else params["steps"]
    n_ck = 3 if fast else 24
    checkpoints = sorted({round(k * n_steps / n_ck) for k in range(n_ck)})
    start = text.index(EXCERPT)
    probe = data[start : start + len(EXCERPT)]  # (L,) the line whose attention maps are drawn
    hist = solve.train(train_data, val_data, w, chars, n_steps, params["batch"], ctx, params["lr"],
                       params["temperature"], set(checkpoints), probe, n_sample=40 if fast else 200)  # fmt: skip
    print(
        f"loss: first batch {hist['train'][0]:.3f} (ln V = {math.log(V):.3f}), "
        f"validation at the end {hist['val'][-1]:.3f}"
    )
    for k in sorted({0, len(hist["at"]) // 3, len(hist["at"]) - 1}):
        first = hist["text"][k].replace("\n", " / ")[:90]
        print(f"  step {hist['at'][k]:4d}  val {hist['val'][k]:.3f}  sample: {first}")

    A = hist["maps"][-1]  # (layers, H, L, L) final model on the probe line
    prev = torch.diagonal(A, offset=-1, dim1=-2, dim2=-1).mean(-1)  # (layers, H) weight on the previous char
    heads_shown = prev.argmax(-1).tolist()
    print(f"attention rows sum to 1: max |Σ_j A_ij − 1| = {(A.sum(-1) - 1).abs().max():.1e}; "
          f"weight above the diagonal: {torch.triu(A, 1).abs().max():.1e}")  # fmt: skip
    for layer, h in enumerate(heads_shown):
        print(f"layer {layer + 1}: head {h + 1} puts {prev[layer, h]:.2f} of its weight on the previous character")
    all_maps = torch.stack(hist["maps"]).numpy()
    return plot.animate(hist, all_maps, EXCERPT, heads_shown, math.log(V), out_dir / "tiny_transformer.gif",
                        hold=1 if fast else 3)  # fmt: skip


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/ai/tiny_transformer/equation.py
r"""GPT → tensor translation: a decoder-only transformer written with einsum.

Indices: n = sequence in the batch, i = query position, j = key position, c = model channel, h = head,
k = channel within a head, v = vocabulary letter.

    X_{nic} = T_{t_{ni} c} + P_{ic}                                      token + position embedding
    Q_{nhik} = \hat X_{nic} W^Q_{hck} + b^Q_{hk}                         (c summed; K and V alike)
    S_{nhij} = Q_{nhik} K_{nhjk} / \sqrt{d_k} + M_{ij}                   (k summed)  M_{ij} = 0 if j ≤ i, −∞ if j > i
    A_{nhij} = \mathrm{softmax}_j S_{nhij}                               each row sums to 1, zero above the diagonal
    X_{nic} \mathrel{+}= A_{nhij} V_{nhjk} W^O_{hkc} + b^O_c             (h, j, k summed)
    X \mathrel{+}= \mathrm{GELU}(\hat X W_1 + c_1) W_2 + c_2             MLP, \hat X = LayerNorm(X)
    z_{niv} = \hat X_{nic} W_{cv} + b_v,   \mathcal L = -\langle \log \mathrm{softmax}_v z_{niv}[t_{n,i+1}] \rangle

The causal mask M makes position i blind to every later character, so one forward pass trains all T next-letter
predictions of a window at once.
"""

import math

import torch


def layer_norm(X: torch.Tensor, g: torch.Tensor, b: torch.Tensor, eps: float = 1e-5) -> torch.Tensor:
    """(X − mean) / std over the channels c of every token, then a learned scale g_c and shift b_c."""
    mu = X.mean(-1, keepdim=True)
    var = X.var(-1, unbiased=False, keepdim=True)
    return g * (X - mu) / torch.sqrt(var + eps) + b


def causal_mask(t: int, device: torch.device) -> torch.Tensor:
    """M_{ij} = 0 for j ≤ i (the past and present) and −∞ for j > i (the future).  (T, T)"""
    future = torch.triu(torch.ones(t, t, dtype=torch.bool, device=device), diagonal=1)
    return torch.zeros(t, t, device=device).masked_fill(future, float("-inf"))


def attention(Q: torch.Tensor, K: torch.Tensor, V: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
    """Causal scaled dot-product attention.  Q, K, V (N, H, T, d_k) → output (N, H, T, d_k) and A (N, H, T, T).

    S_{nhij} = Q_{nhik} K_{nhjk} / √d_k + M_{ij} (k summed); A = softmax_j S; out_{nhik} = A_{nhij} V_{nhjk} (j summed).
    """
    t = Q.shape[-2]
    S = torch.einsum("nhik,nhjk->nhij", Q, K) / math.sqrt(Q.shape[-1])
    A = torch.softmax(S + causal_mask(t, Q.device).to(S.dtype), dim=-1)
    return torch.einsum("nhij,nhjk->nhik", A, V), A


def self_attention(X: torch.Tensor, p: dict) -> tuple[torch.Tensor, torch.Tensor]:
    """Multi-head causal self-attention of normalised tokens X (N, T, d) → update (N, T, d) and A (N, H, T, T)."""
    Q = torch.einsum("nic,hck->nhik", X, p["Wq"]) + p["bq"][:, None, :]  # c summed
    K = torch.einsum("njc,hck->nhjk", X, p["Wk"]) + p["bk"][:, None, :]
    V = torch.einsum("njc,hck->nhjk", X, p["Wv"]) + p["bv"][:, None, :]
    mix, A = attention(Q, K, V)
    return torch.einsum("nhik,hkc->nic", mix, p["Wo"]) + p["bo"], A  # h, k summed: heads merged into d channels


def mlp(X: torch.Tensor, p: dict) -> torch.Tensor:
    """Position-wise d → 4d → d network with a GELU in the middle."""
    return torch.nn.functional.gelu(X @ p["W1"] + p["c1"]) @ p["W2"] + p["c2"]


def block(X: torch.Tensor, p: dict) -> tuple[torch.Tensor, torch.Tensor]:
    """One pre-norm transformer block with residual connections: X + Attn(LN X), then X + MLP(LN X)."""
    update, A = self_attention(layer_norm(X, p["g1"], p["b1"]), p)
    X = X + update
    X = X + mlp(layer_norm(X, p["g2"], p["b2"]), p)
    return X, A


def forward(tokens: torch.Tensor, w: dict) -> tuple[torch.Tensor, torch.Tensor]:
    """Next-character logits z (N, T, V) and every layer's attention (N, layers, H, T, T)."""
    t = tokens.shape[1]
    X = w["tok"][tokens] + w["pos"][:t]  # (N, T, d)
    maps = []
    for p in w["layers"]:
        X, A = block(X, p)
        maps.append(A)
    return layer_norm(X, w["gf"], w["bf"]) @ w["head"] + w["bias"], torch.stack(maps, dim=1)


def cross_entropy(logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
    """Mean −log p(next character) in nats.  logits (N, T, V), targets (N, T)."""
    logp = torch.log_softmax(logits, dim=-1)
    return -torch.gather(logp, -1, targets[..., None]).mean()


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/ai/tiny_transformer/tensors.py
"""Build the concrete tensors: the Shakespeare text as integer tokens, training batches and the GPT weights.

The text (data/shakespeare.txt) is Macbeth and Romeo and Juliet from the public-domain Complete Works in Project
Gutenberg eBook #100, with the eBook header and licence removed and curly quotes turned into ASCII.
Every distinct character is one token (V of them). The weights are plain tensors in a dict (no nn.Module,
no nn.Transformer) so every product in equation.py is an explicit einsum.
"""

import math
from pathlib import Path

import torch

DATA = Path(__file__).parent / "data" / "shakespeare.txt"


def load_text() -> str:
    return DATA.read_text(encoding="ascii")


def encode_text(text: str, device: torch.device) -> tuple[list[str], torch.Tensor]:
    """The sorted character vocabulary (V,) and the whole text as token ids (len(text),) int64."""
    chars = sorted(set(text))
    index = {ch: i for i, ch in enumerate(chars)}
    return chars, torch.tensor([index[ch] for ch in text], dtype=torch.long, device=device)


def decode(tokens: torch.Tensor, chars: list[str]) -> str:
    return "".join(chars[i] for i in tokens.tolist())


def batch(data: torch.Tensor, n: int, ctx: int, gen: torch.Generator) -> tuple[torch.Tensor, torch.Tensor]:
    """n random windows of the text: inputs x (n, T) and targets y (n, T), y = x shifted one character left."""
    start = torch.randint(len(data) - ctx - 1, (n,), generator=gen, device=data.device)
    idx = start[:, None] + torch.arange(ctx, device=data.device)  # (n, T) positions in the text
    return data[idx], data[idx + 1]


def init_weights(v: int, d: int, heads: int, layers: int, ctx: int, seed: int, device: torch.device) -> dict:
    """GPT weights. c = model channel (d), h = head, k = channel within a head (d / heads).

    tok (V, d) token embedding;  pos (T, d) learned position embedding
    per layer:  g1, b1 (d,) LayerNorm;  Wq, Wk, Wv (H, d, d/H) + biases (H, d/H);  Wo (H, d/H, d) + bo (d,)
                g2, b2 (d,) LayerNorm;  W1 (d, 4d) + c1 (4d,);  W2 (4d, d) + c2 (d,)
    gf, bf (d,) final LayerNorm;  head (d, V) + bias (V,) read-out to next-character logits
    The read-out starts tiny, so every logit is ≈ 0 and the first loss is ln V.
    """
    gen = torch.Generator().manual_seed(seed)
    dk = d // heads
    s = 1 / math.sqrt(d)

    def rand(*shape, scale):
        return (scale * torch.randn(*shape, generator=gen)).to(device).requires_grad_(True)

    def const(value, *shape):
        return torch.full(shape, float(value), device=device, requires_grad=True)

    w = {"tok": rand(v, d, scale=0.3), "pos": rand(ctx, d, scale=0.3)}
    w["layers"] = [
        {
            "g1": const(1, d),
            "b1": const(0, d),
            "Wq": rand(heads, d, dk, scale=s),
            "bq": const(0, heads, dk),
            "Wk": rand(heads, d, dk, scale=s),
            "bk": const(0, heads, dk),
            "Wv": rand(heads, d, dk, scale=s),
            "bv": const(0, heads, dk),
            "Wo": rand(heads, dk, d, scale=s / math.sqrt(2 * layers)),
            "bo": const(0, d),
            "g2": const(1, d),
            "b2": const(0, d),
            "W1": rand(d, 4 * d, scale=s),
            "c1": const(0, 4 * d),
            "W2": rand(4 * d, d, scale=0.5 * s / math.sqrt(2 * layers)),
            "c2": const(0, d),
        }  # fmt: skip
        for _ in range(layers)
    ]
    w.update({"gf": const(1, d), "bf": const(0, d), "head": rand(d, v, scale=1e-3), "bias": const(0, v)})
    return w


def parameters(w: dict) -> list[torch.Tensor]:
    """All trainable tensors, for the optimiser."""
    top = [w["tok"], w["pos"], w["gf"], w["bf"], w["head"], w["bias"]]
    return top + [t for layer in w["layers"] for t in layer.values()]


def parameter_count(v: int, d: int, layers: int, ctx: int) -> int:
    """V d (tokens) + T d (positions) + L (12 d² + 13 d) (blocks) + 2 d (final LayerNorm) + d V + V (read-out).

    Per block: Q, K, V, O are 4 d² weights + 4 d biases; the MLP d → 4d → d is 8 d² + 5 d; two LayerNorms 4 d.
    """
    return v * d + ctx * d + layers * (12 * d * d + 13 * d) + 2 * d + d * v + v


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/ai/tiny_transformer/solve.py
"""Train the GPT on next-character prediction with Adam, and sample text from it."""

import math

import torch

from problems._common.progress import steps

from . import equation, tensors


def overfit(x: torch.Tensor, y: torch.Tensor, w: dict, n_steps: int, lr: float) -> list[float]:
    """Adam on one fixed batch: a correct model and gradient memorise it. Returns the loss per step."""
    opt = torch.optim.Adam(tensors.parameters(w), lr=lr)
    losses = []
    for _ in range(n_steps):
        loss = equation.cross_entropy(equation.forward(x, w)[0], y)
        opt.zero_grad()
        loss.backward()
        opt.step()
        losses.append(loss.item())
    return losses


@torch.no_grad()
def sample(w: dict, prompt: torch.Tensor, n_new: int, ctx: int, temperature: float, seed: int) -> torch.Tensor:
    """Generate n_new characters one at a time: feed the last ctx tokens, draw the next from softmax(z / temperature).

    Autoregression is a loop over time by nature: every new character depends on the ones drawn before it.
    """
    gen = torch.Generator(device=prompt.device).manual_seed(seed)
    out = prompt.clone()
    for _ in range(n_new):
        logits, _ = equation.forward(out[None, -ctx:], w)
        p = torch.softmax(logits[0, -1] / temperature, dim=-1)
        out = torch.cat([out, torch.multinomial(p, 1, generator=gen)])
    return out[len(prompt) :]


def train(data, val, w, chars, n_steps, batch, ctx, lr, temperature, checkpoints, probe, n_sample=160, seed=0):
    """AdamW with a short warm-up and cosine decay. At each checkpoint: validation loss and a text sample.

    Returns the train loss per step and, at the checkpoints, the validation loss, the sampled text and the
    attention maps (layers, H, T, T) of the fixed `probe` line.
    """
    gen = torch.Generator(device=data.device).manual_seed(seed)
    val_gen = torch.Generator(device=data.device).manual_seed(seed + 1)
    xv, yv = tensors.batch(val, 64, ctx, val_gen)  # one fixed validation batch
    opt = torch.optim.AdamW(tensors.parameters(w), lr=lr, weight_decay=0.01)
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda t: min(1.0, (t + 1) / 30) * (0.1 + 0.9 * 0.5 * (1 + math.cos(math.pi * t / n_steps)))
    )
    prompt = probe[:1]  # every sample starts from the first letter of the probe line
    hist = {"train": [], "at": [], "val": [], "text": [], "maps": []}

    def checkpoint(t):
        with torch.no_grad():
            hist["val"].append(equation.cross_entropy(equation.forward(xv, w)[0], yv).item())
            hist["maps"].append(equation.forward(probe[None], w)[1][0].cpu())
        hist["at"].append(t)
        hist["text"].append(tensors.decode(sample(w, prompt, n_sample, ctx, temperature, seed=seed), chars))

    for t in steps(n_steps, "training steps"):
        if t in checkpoints:
            checkpoint(t)
        x, y = tensors.batch(data, batch, ctx, gen)
        loss = equation.cross_entropy(equation.forward(x, w)[0], y)
        opt.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(tensors.parameters(w), 1.0)
        opt.step()
        sched.step()
        hist["train"].append(loss.item())
    checkpoint(n_steps)
    return hist


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/ai/tiny_transformer/plot.py
"""Animate training: the loss falls from ln V, the samples turn from gibberish into Shakespeare-ish text, and the
causal attention maps of a fixed line take shape."""

import textwrap
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from problems._common.plotting import ACCENT, save_gif


def _label(ch: str) -> str:
    return {" ": "_", "\n": "\\n"}.get(ch, ch)


def animate(hist: dict, maps: np.ndarray, excerpt: str, heads: list[int], log_v: float, path: Path, hold: int = 3):
    """hist from solve.train; maps (checkpoints, layers, H, T, T) attention on the excerpt; heads[l] shown per layer."""
    at, val = np.asarray(hist["at"]), np.asarray(hist["val"])
    train = np.asarray(hist["train"])
    win = max(1, len(train) // 30)
    smooth = np.convolve(train, np.ones(win) / win, mode="valid")
    smooth_steps = np.arange(len(smooth)) + win - 1
    n_layers = maps.shape[1]
    labels = [_label(ch) for ch in excerpt]
    L = len(excerpt)

    fig = plt.figure(figsize=(9.8, 6.0))
    gs = fig.add_gridspec(2, 1 + n_layers, height_ratios=[1.6, 0.8], width_ratios=[1.25] + [1] * n_layers)
    ax_loss = fig.add_subplot(gs[0, 0])
    ax_text = fig.add_subplot(gs[1, :])
    ax_att = [fig.add_subplot(gs[0, 1 + layer]) for layer in range(n_layers)]

    (train_line,) = ax_loss.plot([], [], color="#93c5fd", lw=1.2, label="train (running mean)")
    (val_line,) = ax_loss.plot([], [], "o-", color=ACCENT[0], ms=3, lw=1.5, label="validation")
    ax_loss.axhline(log_v, color=ACCENT[1], ls="--", lw=1.2, label=f"ln V = {log_v:.2f} (uniform guess)")
    ax_loss.set_xlim(0, at[-1])
    ax_loss.set_ylim(min(val.min(), smooth.min()) - 0.2, log_v + 0.3)
    ax_loss.set_xlabel("training step")
    ax_loss.set_ylabel("next-character loss (nats)")
    ax_loss.legend(fontsize=7, loc="upper right")
    title = ax_loss.set_title("")

    ims = []
    for layer, ax in enumerate(ax_att):
        im = ax.imshow(maps[0, layer, heads[layer]], cmap=plt.get_cmap("Purples", 12), vmin=0, vmax=1)
        ims.append(im)
        ax.set_xticks(range(L), labels, fontsize=6.5, family="monospace")
        ax.set_yticks(range(L), labels, fontsize=6.5, family="monospace")
        ax.tick_params(length=0, pad=1)
        ax.grid(False)
        ax.set_title(f"layer {layer + 1}, head {heads[layer] + 1}: $A_{{ij}}$", fontsize=9)
        ax.set_xlabel("key $j$ (earlier characters)", fontsize=7)
    ax_att[0].set_ylabel("query $i$", fontsize=7)

    ax_text.axis("off")
    body = ax_text.text(0.0, 0.97, "", va="top", ha="left", family="monospace", fontsize=8.5,
                        transform=ax_text.transAxes)  # fmt: skip

    def update(frame: int):
        k = frame // hold
        seen = smooth_steps <= at[k]
        train_line.set_data(smooth_steps[seen], smooth[seen])
        val_line.set_data(at[: k + 1], val[: k + 1])
        title.set_text(f"step {at[k]}: validation loss {val[k]:.2f} nats")
        for layer, im in enumerate(ims):
            im.set_data(maps[k, layer, heads[layer]])
        lines = hist["text"][k].split("\n")[:9]
        wrapped = "\n".join(textwrap.shorten(s, 110, placeholder="…") if len(s) > 110 else s for s in lines)
        body.set_text(f"sample at step {at[k]} (temperature-scaled, same random seed every time):\n\n{wrapped}")

    fig.tight_layout(rect=(0, 0, 1, 0.97))
    return save_gif(fig, update, list(range(len(at) * hold)), path, fps=6)


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    'steps': 600,
    'd_model': 64,
    'heads': 4,
    'layers': 2,
    'context': 64,
    'batch': 32,
    'lr': 0.003,
    'temperature': 0.8
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.ai.tiny_transformer")]:
    del sys.modules[name]
main = importlib.import_module("problems.ai.tiny_transformer.main")
problem_dir = Path("problems/ai/tiny_transformer")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
